# Prompt Length and Complexity Management

## Overview

This tutorial explores techniques for managing prompt length and complexity when working with large language models (LLMs). We'll focus on two key aspects: balancing detail and conciseness in prompts, and strategies for handling long contexts.

## Motivation

Effective prompt engineering often requires finding the right balance between providing enough context for the model to understand the task and keeping prompts concise for efficiency. Additionally, many real-world applications involve processing long documents or complex multi-step tasks, which can exceed the context window of LLMs. Learning to manage these challenges is crucial for building robust AI applications.

## Key Components

1. Balancing detail and conciseness in prompts
2. Strategies for handling long contexts
3. Practical examples using a locally-hosted open-source model (Qwen3-14B) via HuggingFace Transformers

## Method Details

We'll start by examining techniques for crafting prompts that provide sufficient context without unnecessary verbosity. This includes using clear, concise language and leveraging prompt templates for consistency.

Next, we'll explore strategies for handling long contexts, such as:
- Chunking: Breaking long texts into smaller, manageable pieces
- Summarization: Condensing long texts while retaining key information
- Iterative processing: Handling complex tasks through multiple inference calls

Throughout the tutorial, we'll use practical examples to demonstrate these concepts, utilizing a quantized open-source model running locally in Google Colab with HuggingFace Transformers.

## Conclusion

By the end of this tutorial, you'll have a solid understanding of how to manage prompt length and complexity effectively. These skills will enable you to create more efficient and robust AI applications, capable of handling a wide range of text processing tasks.

## Why Context Windows Exist: The O(n²) Problem

Before diving into prompt engineering techniques, let's understand *why* prompt length matters at a fundamental level.

### The Attention Mechanism's Quadratic Cost

Transformer models (including Qwen3-14B) use **self-attention** to understand relationships between tokens. For each token in the input, the model computes an attention score against **every other token**. This means:

- With **n** input tokens, the model computes **n × n = n²** attention scores per attention head
- Qwen3-14B has **48 layers** with **40 attention heads** each — that's 1,920 attention computations, each of size n²
- **Doubling** the input length doesn't just double the work — it **quadruples** the attention computation
- This is why going from a 1,000-token prompt to a 4,000-token prompt is roughly **16× more expensive** in attention alone

This quadratic scaling is the fundamental reason models have context window limits (128K tokens for Qwen3-14B) and why shorter prompts generate responses faster.

In [ ]:
import time

# Demonstrate that longer inputs measurably increase generation time
# We create prompts of increasing length and time the generation

padding_sentence = "The field of artificial intelligence continues to evolve rapidly. "

test_lengths = [100, 500, 1000, 2000]
results = []

for target_tokens in test_lengths:
    # Build a prompt with approximately the target number of tokens
    base_prompt = "Summarize the following text in one sentence:\n\n"
    padding = padding_sentence
    while count_tokens(base_prompt + padding) < target_tokens:
        padding += padding_sentence

    prompt = base_prompt + padding
    actual_tokens = count_tokens(prompt)

    messages = [{"role": "user", "content": prompt}]

    # Time the generation
    start = time.time()
    response = generate(messages, max_new_tokens=50, temperature=0.7)
    elapsed = time.time() - start

    results.append((actual_tokens, elapsed))
    print(f"Input: {actual_tokens:>5} tokens \u2192 Generation time: {elapsed:.2f}s")

print("\n--- Scaling Analysis ---")
if len(results) >= 2:
    base_tokens, base_time = results[0]
    for tokens, t in results[1:]:
        ratio = tokens / base_tokens
        time_ratio = t / base_time
        print(f"{tokens} tokens is {ratio:.1f}x more tokens than {base_tokens}, "
              f"but took {time_ratio:.1f}x longer")
    print("\nNote: The time increase includes both attention (O(n\u00B2)) and other factors.")
    print("In practice, KV-cache and optimizations reduce the raw quadratic impact,")
    print("but the trend is clear: longer inputs = slower generation.")

### The Numbers Behind the Scenes

Let's put concrete numbers to the attention computation for Qwen3-14B:

| Input Length | Attention Ops per Head | × 40 Heads × 48 Layers | Total Attention Ops |
|---|---|---|---|
| 100 tokens | 10,000 | × 1,920 | 19.2 million |
| 1,000 tokens | 1,000,000 | × 1,920 | 1.92 billion |
| 10,000 tokens | 100,000,000 | × 1,920 | 192 billion |
| 128,000 tokens | 16,384,000,000 | × 1,920 | ~31.5 trillion |

This is why **quantization alone isn't enough** for efficiency. Even with 4-bit quantization reducing memory per parameter, the attention computation still scales quadratically with sequence length. Effective prompt engineering — keeping prompts short and information-dense — directly reduces this computational burden.

> **Key Insight**: Every unnecessary token in your prompt doesn't just waste space — it adds a row AND a column to every attention matrix in every head of every layer.

## Setup

First, let's install dependencies, load the model, and define our helper functions for generation and token counting.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def count_tokens(text):
    """Count tokens using the model's tokenizer."""
    return len(tokenizer.encode(text))

print("Setup complete!")

## Tokenization: The True Unit of Prompt Length

When managing prompt length, you need to think in **tokens**, not words or characters. The tokenizer determines how text is broken into tokens, and this mapping is far from uniform. Let's explore how different content types tokenize — and why it matters for prompt budgets.

In [ ]:
# Interactive token counting and analysis tools

def analyze_tokens(text, label="Text"):
    """Analyze tokenization of a text sample."""
    tokens = tokenizer.encode(text)
    token_strings = [tokenizer.decode([t]) for t in tokens]
    words = text.split()
    chars = len(text)

    print(f"[{label}]")
    print(f"  Characters: {chars}, Words: {len(words)}, Tokens: {len(tokens)}")
    print(f"  Ratios \u2014 chars/token: {chars/len(tokens):.1f}, words/token: {len(words)/len(tokens):.2f}")
    print(f"  First 15 tokens: {token_strings[:15]}")
    print()
    return len(tokens)

# Compare different content types
prose = "The quick brown fox jumps over the lazy dog near the river bank on a sunny afternoon."
python_code = 'def calculate_average(numbers): return sum(numbers) / len(numbers) if numbers else 0.0'
numbers = "2024-01-15T14:30:00Z lat=40.7128 lon=-74.0060 temp=72.5F humidity=65%"
url_text = "Visit https://api.example.com/v2/users?page=1&limit=50&sort=created_at&order=desc"
json_text = '{"user_id": 12345, "name": "Alice", "scores": [98, 87, 92], "active": true}'

print("=" * 60)
print("TOKEN ANALYSIS: How different content types tokenize")
print("=" * 60 + "\n")

t_prose = analyze_tokens(prose, "English prose")
t_code = analyze_tokens(python_code, "Python code")
t_numbers = analyze_tokens(numbers, "Numbers/dates/coords")
t_url = analyze_tokens(url_text, "URL with parameters")
t_json = analyze_tokens(json_text, "JSON data")

print("=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"  Prose:   {t_prose:>3} tokens ({len(prose):>3} chars)")
print(f"  Code:    {t_code:>3} tokens ({len(python_code):>3} chars)")
print(f"  Numbers: {t_numbers:>3} tokens ({len(numbers):>3} chars)")
print(f"  URL:     {t_url:>3} tokens ({len(url_text):>3} chars)")
print(f"  JSON:    {t_json:>3} tokens ({len(json_text):>3} chars)")
print()
print("Key takeaway: Code, URLs, and structured data use significantly")
print("more tokens per character than natural language prose.")

### Rule of Thumb for Token Estimation

For Qwen3's tokenizer (based on byte-pair encoding):

| Content Type | Chars per Token | Words per Token |
|---|---|---|
| English prose | ~4.0 | ~0.75 |
| Python code | ~3.0 | ~0.50 |
| URLs / paths | ~2.5 | N/A |
| JSON / structured data | ~2.5 | N/A |
| Numbers / dates | ~2.0 | N/A |

**Practical implications:**
- When estimating prompt length, don't just count words — consider the content type
- Embedding raw JSON, URLs, or code in prompts is especially token-expensive
- If you need to include structured data, consider converting it to natural language descriptions first
- Use `count_tokens()` to measure actual usage rather than guessing

## Information Density: Not All Tokens Are Equal

A key insight for prompt optimization: **not all tokens carry equal information**. In the sentence "The cat sat on the mat", the words "cat", "sat", and "mat" carry most of the meaning, while "The", "on", and "the" are largely predictable filler.

The model's attention mechanism naturally learns this — high-information tokens receive more attention weight. Understanding this lets us compress prompts by removing low-information tokens while preserving meaning.

In [ ]:
# Information density analysis: measure how much each token matters
# by removing it and seeing how much the output changes (ablation study)

test_prompt = "Explain the key differences between supervised and unsupervised machine learning algorithms."
tokens = tokenizer.encode(test_prompt)
token_strings = [tokenizer.decode([t]) for t in tokens]

print(f"Original prompt: \"{test_prompt}\"")
print(f"Tokens ({len(tokens)}): {token_strings}\n")

# Generate a reference output with greedy decoding for reproducibility
ref_messages = [{"role": "user", "content": test_prompt}]
reference_output = generate(ref_messages, max_new_tokens=100, temperature=0.01, do_sample=False)

print(f"Reference output (first 150 chars): {reference_output[:150]}...\n")

# Ablation: remove each token one at a time and measure output change
print("--- Token Ablation Study ---")
print(f"Removing each token and measuring how much the output changes...\n")

impact_scores = []
for i in range(len(tokens)):
    # Reconstruct prompt without this token
    remaining = tokens[:i] + tokens[i+1:]
    ablated_prompt = tokenizer.decode(remaining)

    abl_messages = [{"role": "user", "content": ablated_prompt}]
    ablated_output = generate(abl_messages, max_new_tokens=100, temperature=0.01, do_sample=False)

    # Jaccard similarity on word sets
    ref_words = set(reference_output.lower().split())
    abl_words = set(ablated_output.lower().split())
    if len(ref_words | abl_words) > 0:
        overlap = len(ref_words & abl_words) / len(ref_words | abl_words)
    else:
        overlap = 1.0

    impact = 1.0 - overlap  # Higher = more impact when removed
    impact_scores.append((token_strings[i], impact))

# Sort and display ranked by information content
print("--- Ranked by Information Content ---")
sorted_scores = sorted(impact_scores, key=lambda x: x[1], reverse=True)
for token_str, score in sorted_scores:
    bar = "\u2588" * int(score * 30)
    label = "HIGH info" if score > 0.3 else ("medium" if score > 0.1 else "low info")
    print(f"  {repr(token_str):<20} {score:.2f}  {bar}  {label}")

In [ ]:
# Prompt compression: remove low-information words, preserve meaning

original = """Please provide a detailed and comprehensive explanation of the main
differences between supervised learning and unsupervised learning in the field
of machine learning. Include specific examples of each type and discuss the
practical applications where each approach is most commonly used."""

# Compressed: same core meaning, telegraphic style
compressed = """Explain differences between supervised vs unsupervised learning.
Examples and practical applications of each."""

orig_tokens = count_tokens(original)
comp_tokens = count_tokens(compressed)
savings = (1 - comp_tokens / orig_tokens) * 100

print(f"Original prompt:    {orig_tokens} tokens")
print(f"Compressed prompt:  {comp_tokens} tokens")
print(f"Token savings:      {savings:.0f}%\n")

# Compare outputs
print("=" * 60)
print("ORIGINAL PROMPT OUTPUT:")
print("=" * 60)
orig_messages = [{"role": "user", "content": original}]
orig_response = generate(orig_messages, max_new_tokens=300)
print(orig_response[:500])
print(f"\n[Response tokens: {count_tokens(orig_response)}]")

print("\n" + "=" * 60)
print("COMPRESSED PROMPT OUTPUT:")
print("=" * 60)
comp_messages = [{"role": "user", "content": compressed}]
comp_response = generate(comp_messages, max_new_tokens=300)
print(comp_response[:500])
print(f"\n[Response tokens: {count_tokens(comp_response)}]")

print("\n" + "=" * 60)
print("VERDICT")
print("=" * 60)
print(f"The compressed prompt used {savings:.0f}% fewer input tokens")
print(f"while producing a response of similar quality and coverage.")
print(f"This is the core principle: keep high-information tokens, remove filler.")

### The Principle of Prompt Compression

The ablation study above demonstrates a fundamental principle of prompt optimization:

1. **Function words** ("the", "a", "of", "in", "please") carry minimal information — removing them barely changes the output
2. **Content words** ("supervised", "unsupervised", "differences", "learning") are critical — removing them fundamentally alters or breaks the output
3. **Instruction modifiers** ("detailed", "comprehensive", "specific") have moderate impact — they shape the response style but the core meaning survives without them

**Practical compression rules:**
- Remove courtesy words ("please", "kindly")
- Remove redundant adjectives ("detailed and comprehensive" → just state what you want)
- Use telegraphic style for instructions ("Explain X vs Y" instead of "Please provide an explanation of the differences between X and Y")
- Keep all domain-specific terms and proper nouns — these are your highest-information tokens

## Balancing Detail and Conciseness

Let's start by examining how to balance detail and conciseness in prompts. We'll compare responses from a detailed prompt and a concise prompt, and use token counting to understand the cost of each approach.

In [ ]:
topic = "artificial intelligence"

# Detailed prompt
detailed_template = """Please provide a comprehensive explanation of {topic}. Include its definition,
historical context, key components, practical applications, and any relevant examples.
Also, discuss any controversies or debates surrounding the topic, and mention potential
future developments or trends."""

# Concise prompt
concise_template = "Briefly explain {topic} and its main importance."

detailed_prompt_text = detailed_template.format(topic=topic)
concise_prompt_text = concise_template.format(topic=topic)

print(f"Detailed prompt tokens: {count_tokens(detailed_prompt_text)}")
print(f"Concise prompt tokens:  {count_tokens(concise_prompt_text)}")

detailed_messages = [{"role": "user", "content": detailed_prompt_text}]
concise_messages = [{"role": "user", "content": concise_prompt_text}]

print("\nDetailed response:")
detailed_response = generate(detailed_messages, max_new_tokens=1024)
print(detailed_response)
print(f"\n[Detailed response tokens: {count_tokens(detailed_response)}]")

print("\n" + "="*60)
print("\nConcise response:")
concise_response = generate(concise_messages, max_new_tokens=256)
print(concise_response)
print(f"\n[Concise response tokens: {count_tokens(concise_response)}]")

### Analysis of Prompt Balance

Let's analyze the differences between the detailed and concise prompts, and discuss strategies for finding the right balance.

In [ ]:
analysis_template = """Compare the following two responses on {topic}:

Detailed response:
{detailed_response}

Concise response:
{concise_response}

Analyze the differences in terms of:
1. Information coverage
2. Clarity and focus
3. Potential use cases for each type of response

Then, suggest strategies for balancing detail and conciseness in prompts."""

analysis_prompt_text = analysis_template.format(
    topic=topic,
    detailed_response=detailed_response,
    concise_response=concise_response
)

print(f"Analysis prompt tokens: {count_tokens(analysis_prompt_text)}")
print(f"(This shows how combining prior outputs increases prompt length)\n")

messages = [{"role": "user", "content": analysis_prompt_text}]
analysis = generate(messages, max_new_tokens=1024)
print(analysis)

## Strategies for Handling Long Contexts

Now, let's explore strategies for handling long contexts, which often exceed the token limits of language models.

### 1. Chunking

Chunking involves breaking long texts into smaller, manageable pieces. We use a custom Python text splitter that tries to break at sentence boundaries for cleaner chunks.

In [ ]:
def split_text(text, chunk_size=1000, chunk_overlap=200):
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        # Try to break at a sentence boundary
        if end < len(text):
            last_period = text.rfind('.', start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        chunks.append(text[start:end].strip())
        start = end - chunk_overlap
    return chunks


long_text = """
Artificial intelligence (AI) is a branch of computer science that aims to create intelligent machines that can simulate human cognitive processes.
The field of AI has a rich history dating back to the 1950s, with key milestones such as the development of the first neural networks and expert systems.
AI encompasses a wide range of subfields, including machine learning, natural language processing, computer vision, and robotics.
Practical applications of AI include speech recognition, image classification, autonomous vehicles, and medical diagnosis.
AI has the potential to revolutionize many industries, from healthcare and finance to transportation and entertainment.
However, there are ongoing debates and controversies surrounding AI, such as concerns about job displacement, bias in algorithms, and the ethical implications of autonomous systems.
Looking ahead, the future of AI holds promise for advancements in areas like explainable AI, AI ethics, and human-AI collaboration.
The intersection of AI with other technologies like blockchain, quantum computing, and biotechnology will likely shape the future of the field.
But as AI continues to evolve, it is essential to consider the societal impact and ethical implications of these technologies.
One of the key challenges for AI researchers and developers is to strike a balance between innovation and responsibility, ensuring that AI benefits society as
a whole while minimizing potential risks.
If managed effectively, AI has the potential to transform our world in ways we can only begin to imagine.
Though the future of AI is uncertain, one thing is clear: the impact of artificial intelligence will be profound and far-reaching.
"""

print(f"Original text length: {len(long_text)} characters, {count_tokens(long_text)} tokens")

chunks = split_text(long_text, chunk_size=1000, chunk_overlap=200)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}: {len(chunk)} chars, {count_tokens(chunk)} tokens")
print(f"\nFirst chunk preview: {chunks[0][:200]}...")

### Chunking Strategies Compared

The `split_text()` function above uses a character-based approach with sentence boundary detection. But there are multiple chunking strategies, each with different trade-offs. Let's implement and compare them side by side.

In [ ]:
import re

# Define multiple chunking strategies

def chunk_fixed_chars(text, size=500):
    """Strategy 1: Naive fixed character count \u2014 splits mid-word, mid-sentence."""
    chunks = []
    for i in range(0, len(text), size):
        chunks.append(text[i:i+size].strip())
    return [c for c in chunks if c]

def chunk_by_sentences(text, max_tokens=150, overlap_sentences=1):
    """Strategy 2: Sentence boundary chunking with overlap."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], []
    current_tokens = 0

    for sent in sentences:
        sent_tokens = count_tokens(sent)
        if current_tokens + sent_tokens > max_tokens and current:
            chunks.append(' '.join(current))
            # Keep last N sentences for overlap
            current = current[-overlap_sentences:] if overlap_sentences > 0 else []
            current_tokens = sum(count_tokens(s) for s in current)
        current.append(sent)
        current_tokens += sent_tokens

    if current:
        chunks.append(' '.join(current))
    return chunks

def chunk_by_paragraphs(text, max_tokens=300):
    """Strategy 3: Paragraph boundary chunking."""
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]

    chunks, current = [], []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = count_tokens(para)
        if current_tokens + para_tokens > max_tokens and current:
            chunks.append('\n'.join(current))
            current = []
            current_tokens = 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append('\n'.join(current))
    return chunks

def chunk_by_tokens(text, max_tokens=150, overlap_tokens=20):
    """Strategy 4: Exact token-count chunking."""
    all_tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(all_tokens):
        end = min(start + max_tokens, len(all_tokens))
        chunk_tokens = all_tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start = end - overlap_tokens if end < len(all_tokens) else end
    return chunks


# Compare all strategies on our sample text
sample = long_text.strip()

strategies = [
    ("Fixed chars (500)", chunk_fixed_chars(sample, size=500)),
    ("Sentence boundary", chunk_by_sentences(sample, max_tokens=150, overlap_sentences=1)),
    ("Paragraph boundary", chunk_by_paragraphs(sample, max_tokens=200)),
    ("Token-count (150)", chunk_by_tokens(sample, max_tokens=150, overlap_tokens=20)),
]

for name, chunks in strategies:
    print(f"\n{'=' * 60}")
    print(f"Strategy: {name} \u2192 {len(chunks)} chunks")
    print('=' * 60)
    for i, chunk in enumerate(chunks):
        tok_count = count_tokens(chunk)
        preview_start = chunk[:50].replace('\n', ' ')
        print(f"  Chunk {i+1}: {tok_count:>4} tokens | \"{preview_start}...\"")

    # Show boundary problem for fixed-char chunking
    if "Fixed" in name and len(chunks) > 1:
        boundary = chunks[0][-30:] + " | " + chunks[1][:30]
        print(f"\n  \u26a0 Boundary cut: \"...{boundary}...\"")
        print(f"     (May split mid-word or mid-sentence!)")

print("\n" + "=" * 60)
print("SUMMARY: Which strategy to use?")
print("=" * 60)
print("  Fixed chars:    Fast but breaks words/sentences \u2014 avoid for LLM input")
print("  Sentence:       Best balance of coherence and control \u2014 recommended")
print("  Paragraph:      Best semantic units but uneven chunk sizes")
print("  Token-count:    Most precise length control for tight context budgets")

### Choosing a Chunking Strategy

| Strategy | Coherence | Size Control | Speed | Best For |
|---|---|---|---|---|
| Fixed character | ❌ Poor | ✅ Exact | ✅ Fast | Never use for LLM input |
| Sentence boundary | ✅ Good | ⚠️ Approximate | ✅ Fast | General-purpose chunking |
| Paragraph boundary | ✅ Best | ❌ Uneven | ✅ Fast | Well-structured documents |
| Token-count | ⚠️ May split words | ✅ Exact tokens | ⚠️ Slower | Tight context budgets |

**Recommendation**: Sentence-boundary chunking with 1–2 sentence overlap is the best default. It preserves coherent units of meaning while giving reasonable control over chunk sizes. The overlap ensures no information is lost at boundaries.

The existing `split_text()` function uses a hybrid approach — character-based with sentence-boundary fallback — which is a practical middle ground.

### 2. Summarization

Summarization can be used to condense long texts while retaining key information. We use a manual map-reduce approach: summarize each chunk individually, then combine the summaries into a final result.

In [ ]:
def map_reduce_summarize(text, chunk_size=2000):
    """Summarize long text using map-reduce approach."""
    chunks = split_text(text, chunk_size=chunk_size)
    print(f"Split into {len(chunks)} chunks ({count_tokens(text)} total tokens)")

    # Map: summarize each chunk
    summaries = []
    for i, chunk in enumerate(chunks):
        messages = [
            {"role": "system", "content": "Summarize the following text concisely."},
            {"role": "user", "content": chunk}
        ]
        summary = generate(messages, max_new_tokens=256)
        summaries.append(summary)
        print(f"Chunk {i+1}/{len(chunks)}: {count_tokens(chunk)} tokens \u2192 {count_tokens(summary)} tokens")

    # Reduce: combine summaries into final summary
    combined = "\n\n".join(summaries)
    messages = [
        {"role": "system", "content": "Combine these summaries into a single coherent summary."},
        {"role": "user", "content": combined}
    ]
    final = generate(messages, max_new_tokens=512)
    print(f"\nFinal summary: {count_tokens(final)} tokens")
    return final


summary = map_reduce_summarize(long_text)
print("\nSummary:")
print(summary)

### 3. Iterative Processing

For complex tasks that require multiple steps, we can use iterative processing. Each step feeds its output as context into the next inference call. Token counting helps us monitor the growing context size.

In [ ]:
def iterative_analysis(text, steps):
    """
    Perform iterative analysis on a given text.

    Args:
        text (str): The text to analyze.
        steps (list): List of analysis steps to perform.

    Returns:
        str: The final analysis result.
    """
    result = text
    for i, step in enumerate(steps):
        prompt_text = f"Analyze the following text. {step}\n\nText: {result}\n\nAnalysis:"
        print(f"Step {i+1}/{len(steps)}: \"{step}\"")
        print(f"  Input tokens: {count_tokens(prompt_text)}")
        messages = [{"role": "user", "content": prompt_text}]
        result = generate(messages, max_new_tokens=512)
        print(f"  Output tokens: {count_tokens(result)}")
    return result


analysis_steps = [
    "Identify the main topics discussed.",
    "Summarize the key points for each topic.",
    "Provide a brief conclusion based on the analysis."
]

final_analysis = iterative_analysis(long_text, analysis_steps)
print("\nFinal Analysis:")
print(final_analysis)

### Comparing Long Context Strategies

We've seen individual strategies (chunking, summarization, iterative processing) above. Now let's directly compare three approaches for processing the same long document, measuring both output quality and total token efficiency.

In [ ]:
import time

# Compare three strategies for processing a long document

print("=" * 60)
print("STRATEGY COMPARISON: Processing a Long Document")
print("=" * 60)

doc = long_text.strip()
doc_tokens = count_tokens(doc)
print(f"Document: {doc_tokens} tokens\n")

# Strategy A: Stuff everything into one prompt
print("-" * 60)
print("Strategy A: Single-prompt (stuff everything in)")
print("-" * 60)
start_a = time.time()
stuff_prompt = f"Provide a comprehensive summary of the following text:\n\n{doc}"
stuff_tokens = count_tokens(stuff_prompt)
messages_a = [{"role": "user", "content": stuff_prompt}]
result_a = generate(messages_a, max_new_tokens=256)
time_a = time.time() - start_a
print(f"  Input tokens:  {stuff_tokens}")
print(f"  Output tokens: {count_tokens(result_a)}")
print(f"  Time: {time_a:.2f}s")
print(f"  Result: {result_a[:300]}...\n")

# Strategy B: Map-reduce
print("-" * 60)
print("Strategy B: Map-Reduce (summarize chunks, then combine)")
print("-" * 60)
start_b = time.time()
total_input_b = 0
chunks_b = split_text(doc, chunk_size=500, chunk_overlap=50)
chunk_summaries = []
for i, chunk in enumerate(chunks_b):
    msg = [
        {"role": "system", "content": "Summarize this text in 2-3 sentences."},
        {"role": "user", "content": chunk}
    ]
    s = generate(msg, max_new_tokens=100)
    chunk_summaries.append(s)
    total_input_b += count_tokens(chunk)

combined = "\n".join(chunk_summaries)
combine_msg = [
    {"role": "system", "content": "Combine these summaries into one coherent summary."},
    {"role": "user", "content": combined}
]
total_input_b += count_tokens(combined)
result_b = generate(combine_msg, max_new_tokens=256)
time_b = time.time() - start_b
print(f"  Chunks: {len(chunks_b)}, Total input tokens: {total_input_b}")
print(f"  Output tokens: {count_tokens(result_b)}")
print(f"  Time: {time_b:.2f}s")
print(f"  Result: {result_b[:300]}...\n")

# Strategy C: Iterative refinement
print("-" * 60)
print("Strategy C: Iterative Refinement (running summary)")
print("-" * 60)
start_c = time.time()
total_input_c = 0
chunks_c = split_text(doc, chunk_size=500, chunk_overlap=50)
running_summary = ""
for i, chunk in enumerate(chunks_c):
    if running_summary:
        prompt_c = f"Current summary:\n{running_summary}\n\nNew text:\n{chunk}\n\nUpdate the summary to incorporate the new information."
    else:
        prompt_c = f"Summarize this text:\n{chunk}"
    total_input_c += count_tokens(prompt_c)
    msg = [{"role": "user", "content": prompt_c}]
    running_summary = generate(msg, max_new_tokens=200)

result_c = running_summary
time_c = time.time() - start_c
print(f"  Chunks: {len(chunks_c)}, Total input tokens: {total_input_c}")
print(f"  Output tokens: {count_tokens(result_c)}")
print(f"  Time: {time_c:.2f}s")
print(f"  Result: {result_c[:300]}...\n")

# Comparison table
print("=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"  {'Strategy':<25} {'Input Tokens':<15} {'Time':<10} {'LLM Calls'}")
print(f"  {'-'*25} {'-'*14} {'-'*9} {'-'*9}")
print(f"  {'A: Single prompt':<25} {stuff_tokens:<15} {time_a:<10.2f} 1")
print(f"  {'B: Map-Reduce':<25} {total_input_b:<15} {time_b:<10.2f} {len(chunks_b) + 1}")
print(f"  {'C: Iterative':<25} {total_input_c:<15} {time_c:<10.2f} {len(chunks_c)}")

### When to Use Each Strategy

| Strategy | Pros | Cons | Best For |
|---|---|---|---|
| **Single prompt** | Simplest, model sees full context | Fails if text exceeds context window | Short-to-medium documents |
| **Map-reduce** | Parallelizable, handles any length | Loses cross-chunk relationships | Long documents, batch processing |
| **Iterative refinement** | Preserves running context | Sequential (slow), summary may drift | Narrative documents, evolving analysis |

**Default recommendation:**
- If the document fits in the context window → use **single prompt** (Strategy A)
- If it doesn't fit → use **map-reduce** (Strategy B) as the default
- Use **iterative refinement** (Strategy C) only when cross-chunk context is critical and you can't fit everything in one prompt

## Practical Tips for Managing Prompt Length and Complexity

Let's conclude with some practical tips for managing prompt length and complexity when working with local open-source models.

In [ ]:
tips_prompt = """Based on the examples and strategies we've explored for managing prompt length and complexity,
provide a list of 5 practical tips for developers working with large language models.
Each tip should be concise and actionable."""

print(f"Prompt tokens: {count_tokens(tips_prompt)}")

messages = [{"role": "user", "content": tips_prompt}]
tips = generate(messages, max_new_tokens=512)
print(f"Response tokens: {count_tokens(tips)}\n")
print(tips)